In [1]:
from __future__ import division 
from collections import Counter 
from collections import defaultdict

# Finding Key Connectors

In [2]:
users = [
    { "id": 0, "name": "Hero" },
    { "id": 1, "name": "Dunn" },
    { "id": 2, "name": "Sue" },
    { "id": 3, "name": "Chi" },
    { "id": 4, "name": "Thor" },
    { "id": 5, "name": "Clive" },
    { "id": 6, "name": "Hicks" },
    { "id": 7, "name": "Devin" },
    { "id": 8, "name": "Kate" },
    { "id": 9, "name": "Klein" }
]

In [3]:
friendships = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4),
                (4, 5), (5, 6), (5, 7), (6, 8), (7, 8), (8, 9)]

In [4]:
for user in users:
    user["friends"] = []

In [5]:
for i, j in friendships:
    users[i]["friends"].append(users[j])  # add j as a friend of i
    users[j]["friends"].append(users[i])  # add i as a friend of j

In [6]:
def number_of_friends(user):
    """How many friends does _user_ have?"""
    return len(user["friends"])

In [7]:
total_connections = sum(number_of_friends(user) for user in users)
total_connections

24

In [8]:
num_users = len(users)
avg_connections = total_connections / num_users
avg_connections

2.4

In [9]:
num_friends_by_id = [(user["id"], number_of_friends(user)) for user in users]

In [10]:
# each pair is (user_id, num_friends)
sorted(num_friends_by_id, key=lambda id_and_friends: id_and_friends[1], reverse=True)

[(1, 3),
 (2, 3),
 (3, 3),
 (5, 3),
 (8, 3),
 (0, 2),
 (4, 2),
 (6, 2),
 (7, 2),
 (9, 1)]

# Data Scientists You May Know

In [11]:
def friends_of_friend_ids_bad(user):
    # "foaf" is short for "friend of a friend"
    return [foaf["id"]
            for friend in user["friends"] # for each of user's friends
            for foaf in friend["friends"]] # get each of _their_ friends

In [12]:
friends_of_friend_ids_bad(users[0])


[0, 2, 3, 0, 1, 3]

In [13]:
def not_the_same(user, other_user):
    """Two users are not the same if they have different ids."""
    return user["id"] != other_user["id"]

In [14]:
def not_friends(user, other_user):
    """Other_user is not a friend if he's not in user["friends"]."""
    return all(not_the_same(friend, other_user)
               for friend in user["friends"])

In [15]:
def friends_of_friend_ids(user):
    return Counter(foaf["id"]
                    for friend in user["friends"] # for each of my friends
                    for foaf in friend["friends"] # count *their* friends
                    if not_the_same(user, foaf) # who aren't me
                    and not_friends(user, foaf)) # and aren't my friends

In [16]:
print(friends_of_friend_ids(users[3])) 

Counter({0: 2, 5: 1})


In [17]:
interests = [
    (0, "Hadoop"), (0, "Big Data"), (0, "HBase"), (0, "Java"),
    (0, "Spark"), (0, "Storm"), (0, "Cassandra"),
    (1, "NoSQL"), (1, "MongoDB"), (1, "Cassandra"), (1, "HBase"),
    (1, "Postgres"), (2, "Python"), (2, "scikit-learn"), (2, "scipy"),
    (2, "numpy"), (2, "statsmodels"), (2, "pandas"), (3, "R"), (3, "Python"),
    (3, "statistics"), (3, "regression"), (3, "probability"),
    (4, "machine learning"), (4, "regression"), (4, "decision trees"),
    (4, "libsvm"), (5, "Python"), (5, "R"), (5, "Java"), (5, "C++"),
    (5, "Haskell"), (5, "programming languages"), (6, "statistics"),
    (6, "probability"), (6, "mathematics"), (6, "theory"),
    (7, "machine learning"), (7, "scikit-learn"), (7, "Mahout"),
    (7, "neural networks"), (8, "neural networks"), (8, "deep learning"),
    (8, "Big Data"), (8, "artificial intelligence"), (9, "Hadoop"),
    (9, "Java"), (9, "MapReduce"), (9, "Big Data")
]

In [18]:
def data_scientists_who_like(target_interest):
    return [user_id
            for user_id, user_interest in interests
            if user_interest == target_interest]

In [19]:
# keys are interests, values are lists of user_ids with that interest
user_ids_by_interest = defaultdict(list)

In [20]:
for user_id, interest in interests:
    user_ids_by_interest[interest].append(user_id)

In [21]:
# keys are user_ids, values are lists of interests for that user_id
interests_by_user_id = defaultdict(list)

In [22]:
for user_id, interest in interests:
    interests_by_user_id[user_id].append(interest)

In [23]:
def most_common_interests_with(user):
    return Counter(interested_user_id
        for interest in interests_by_user_id[user["id"]]
        for interested_user_id in user_ids_by_interest[interest]
        if interested_user_id != user["id"])

In [24]:
most_common_interests_with(users[0])

Counter({9: 3, 1: 2, 8: 1, 5: 1})

# Salaries and Experience

In [25]:
salaries_and_tenures = [(83000, 8.7), (88000, 8.1),
                        (48000, 0.7), (76000, 6),
                        (69000, 6.5), (76000, 7.5),
                        (60000, 2.5), (83000, 10),
                        (48000, 1.9), (63000, 4.2)]

In [26]:
# keys are years, values are lists of the salaries for each tenure
salary_by_tenure = defaultdict(list)

In [27]:
for salary, tenure in salaries_and_tenures:
    salary_by_tenure[tenure].append(salary)
    
salary_by_tenure

defaultdict(list,
            {8.7: [83000],
             8.1: [88000],
             0.7: [48000],
             6: [76000],
             6.5: [69000],
             7.5: [76000],
             2.5: [60000],
             10: [83000],
             1.9: [48000],
             4.2: [63000]})

In [28]:
# keys are years, each value is average salary for that tenure
average_salary_by_tenure = {
        tenure : sum(salaries) / len(salaries)
        for tenure, salaries in salary_by_tenure.items()
    }

average_salary_by_tenure

{8.7: 83000.0,
 8.1: 88000.0,
 0.7: 48000.0,
 6: 76000.0,
 6.5: 69000.0,
 7.5: 76000.0,
 2.5: 60000.0,
 10: 83000.0,
 1.9: 48000.0,
 4.2: 63000.0}

In [29]:
def tenure_bucket(tenure):
    if tenure < 2:
        return "less than two"
    elif tenure < 5:
        return "between two and five"
    else:
        return "more than five"

In [32]:
# keys are tenure buckets, values are lists of salaries for that bucket
salary_by_tenure_bucket = defaultdict(list)

for salary, tenure in salaries_and_tenures:
    bucket = tenure_bucket(tenure)
    salary_by_tenure_bucket[bucket].append(salary)
    
    
salary_by_tenure_bucket

defaultdict(list,
            {'more than five': [83000, 88000, 76000, 69000, 76000, 83000],
             'less than two': [48000, 48000],
             'between two and five': [60000, 63000]})

In [35]:
# keys are tenure buckets, values are average salary for that bucket
average_salary_by_bucket = {
    tenure_bucket : sum(salaries) / len(salaries)
    for tenure_bucket, salaries in salary_by_tenure_bucket.items()
}
average_salary_by_bucket

{'more than five': 79166.66666666667,
 'less than two': 48000.0,
 'between two and five': 61500.0}

# paid accounts


In [36]:
def predict_paid_or_unpaid(years_experience):
    if years_experience < 3.0:
        return "paid"
    elif years_experience < 8.5:
        return "unpaid"
    else:
        return "paid"